# Similarity-aware Label Smoothing

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders
import pandas as pd
from models import CifarResNET18, CifarDenseNET121, CifarWideResNET2810
from metrics import top_label_ece, nll_loss
import random
import os
from scipy.stats import spearmanr, wilcoxon



/Users/nihar/Desktop/Research/Similarity-Aware Label-Smoothing/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def upper_tri_vec(D):
    """Vectorize upper triangle (excluding diagonal)."""
    idx = np.triu_indices_from(D, k=1)
    return D[idx]

def geometry_spearman(D1, D2):
    v1 = upper_tri_vec(D1)
    v2 = upper_tri_vec(D2)
    rho, _ = spearmanr(v1, v2)
    return rho

def relative_l2_change(D1, D2):
    v1 = upper_tri_vec(D1)
    v2 = upper_tri_vec(D2)
    return np.linalg.norm(v1 - v2) / np.linalg.norm(v1)

def permutation_test_geometry(D1, D2, n_perm=10000):
    v1 = upper_tri_vec(D1)
    v2 = upper_tri_vec(D2)

    observed = np.linalg.norm(v1 - v2)
    diffs = []

    for _ in range(n_perm):
        perm = np.random.permutation(len(v1))
        diffs.append(np.linalg.norm(v1 - v2[perm]))

    diffs = np.array(diffs)
    p_value = (np.sum(diffs >= observed) + 1) / (n_perm + 1)

    return observed, diffs.mean(), diffs.std(), p_value



In [27]:
path = f"scores/05_cifar10_latent_distances.xlsx"
D_05 = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

path = f"scores/1_cifar10_latent_distances.xlsx"
D_1 = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

path = f"scores/4_cifar10_latent_distances.xlsx"
D_4 = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)


pairs = [
    ("0.5", "1", D_05, D_1),
    ("1", "4", D_1, D_4),
    ("0.5", "4", D_05, D_4),
]

for a, b, Da, Db in pairs:
    rho = geometry_spearman(Da, Db)
    delta = relative_l2_change(Da, Db)
    obs, mu, std, p = permutation_test_geometry(Da, Db)

    print(f"β {a} → {b}")
    print(f"  Spearman ρ      = {rho:.3f}")
    print(f"  Relative change = {delta:.3f}")
    print(f"  Permutation p   = {p:.4g}\n")


β 0.5 → 1
  Spearman ρ      = 0.983
  Relative change = 0.048
  Permutation p   = 1

β 1 → 4
  Spearman ρ      = 0.985
  Relative change = 0.115
  Permutation p   = 1

β 0.5 → 4
  Spearman ρ      = 0.989
  Relative change = 0.148
  Permutation p   = 1



In [34]:
path = f"scores/4_cifar100_latent_distances.xlsx"
dists = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

topk_values, topk_indices = torch.topk(-dists[[i for i in range(100)]], 6, dim=1)
[clas.tolist() for clas in topk_indices]

indices = [1, 2, 3]
for j in topk_indices:
    selected_classes = [train_loader.dataset.classes[i] for i in j]
    print(selected_classes)

['apple', 'sweet_pepper', 'wardrobe', 'crab', 'lobster', 'orange']
['aquarium_fish', 'mushroom', 'fox', 'possum', 'lion', 'man']
['baby', 'girl', 'boy', 'table', 'man', 'cup']
['bear', 'beaver', 'otter', 'elephant', 'seal', 'skunk']
['beaver', 'bear', 'otter', 'elephant', 'shrew', 'squirrel']
['bed', 'couch', 'table', 'bicycle', 'clock', 'bowl']
['bee', 'butterfly', 'dinosaur', 'beetle', 'tiger', 'cattle']
['beetle', 'elephant', 'bear', 'otter', 'shrew', 'dinosaur']
['bicycle', 'worm', 'snake', 'keyboard', 'can', 'squirrel']
['bottle', 'can', 'flatfish', 'dinosaur', 'bowl', 'lamp']
['bowl', 'cup', 'flatfish', 'snake', 'can', 'lamp']
['boy', 'girl', 'man', 'woman', 'baby', 'table']
['bridge', 'train', 'house', 'streetcar', 'mountain', 'skyscraper']
['bus', 'streetcar', 'television', 'pickup_truck', 'flatfish', 'train']
['butterfly', 'bee', 'cattle', 'bear', 'tiger', 'snail']
['camel', 'dinosaur', 'cattle', 'tractor', 'elephant', 'crab']
['can', 'bowl', 'dinosaur', 'lamp', 'bicycle', 'cl

## Hyperparams

In [12]:
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [3]:
dataset = "cifar100"
batch_size = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 0.1
epochs = 12
num_classes = int(dataset.removeprefix("cifar"))
# epsilon = 0.01

## Training Utils

In [4]:
def accuracy(model, loader, k = (1, 5)):
    model.eval()
    correct = {key:0 for key in k}
    total = 0

    maxk = max(k)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)

            _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)

            for key in k:
                correct[key] += (pred[:, :key] == y.view(-1, 1)).any(dim=1).sum().item()
            total += y.size(0)

    return {key: correct[key] / total for key in k}

### Label Smoothing

In [5]:
path = f"/content/drive/MyDrive/Code/{dataset}_latent_distances.xlsx"
dists = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

def scores(y, k=5, alpha=10.0):
    class_dists = dists[y]

    mask = torch.eye(class_dists.shape[1], device=device)[y]
    class_dists = class_dists.masked_fill(mask.bool(), float('inf'))
    sims = F.softmax(torch.exp(-class_dists), dim=1)
    sims.scatter_(1, y.unsqueeze(1), 0.0)

    sims = sims ** alpha

    result = sims
    topk_values, topk_indices = torch.topk(sims, k, dim=1)
    result = torch.zeros_like(sims)
    result.scatter_(1, topk_indices, topk_values)

    result = result / (result.sum(dim=1, keepdim=True))

    return result

def similarity_aware_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return (1 - epsilon) * y_onehot + epsilon * scores(y)


## Training Loop

In [6]:
def train(model, train_loader, test_loader, classwise_target, optimizer=None, scheduler=None, num_classes=100, epochs=10, epsilon=0.1):

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device), y.to(device)

            targets = classwise_target[y]

            optimizer.zero_grad()
            logits = model(x)
            loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()


        test_acc = accuracy(model, test_loader)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")


## RUN Experiments

In [7]:
# classwise_target = similarity_aware_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
# print(classwise_target)
# print(classwise_target.shape)

# classwise_target = torch.eye(n=num_classes, device=device)
# print(classwise_target)
# print(classwise_target.shape)


In [11]:
train_loader, test_loader = get_data_loaders(dataset=dataset)



100%|██████████| 169M/169M [00:03<00:00, 53.1MB/s] 
/Users/nihar/Desktop/Research/Similarity-Aware Label-Smoothing/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [9]:
for epsilon in [0.005, 0.02, 0.1, 0.2]:
  classwise_target = uniform_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
  print(classwise_target)

  row_sums = torch.sum(classwise_target, dim=1)
  assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-10), "Rows do not sum to 1"

  model = CifarResNET18(num_classes=num_classes).to(device)
  optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
      optimizer,
      T_max=200
  )

  train(
      model=model,
      train_loader=train_loader,
      test_loader=test_loader,
      classwise_target=classwise_target,
      optimizer=optimizer,
      scheduler=scheduler,
      num_classes=num_classes,
      epochs=200,
      epsilon=epsilon,
  )

  logits_list = []
  labels_list = []

  model.eval()
  with torch.no_grad():
      for x, y in test_loader:
          x = x.to(device)
          y = y.to(device)

          logits = model(x)

          logits_list.append(logits)
          labels_list.append(y)

  logits_all = torch.cat(logits_list, dim=0)
  labels_all = torch.cat(labels_list, dim=0)
  print()
  print("ECE:", top_label_ece(logits_all, labels_all))
  print("NLL:", nll_loss(logits_all, labels_all))
  test_acc = accuracy(model, test_loader)
  print(f"Top-1 Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")
  print()
  PATH = f"ls_{'0'+str(epsilon).removeprefix("0.")}_{seed}.pth"

  torch.save(model.state_dict(), PATH)
  folder_path = '/content/drive/My Drive/MyModels'
  os.makedirs(folder_path, exist_ok=True) # Creates the directory if it doesn't exist

  # Define the full path for your .pth file
  save_path = os.path.join(folder_path, PATH)

  # Save the model state dictionary
  torch.save(model.state_dict(), save_path)
  print(f"Model saved successfully to {save_path}")
  print()

from google.colab import runtime
runtime.unassign()



tensor([[9.9500e-01, 5.0505e-05, 5.0505e-05,  ..., 5.0505e-05, 5.0505e-05,
         5.0505e-05],
        [5.0505e-05, 9.9500e-01, 5.0505e-05,  ..., 5.0505e-05, 5.0505e-05,
         5.0505e-05],
        [5.0505e-05, 5.0505e-05, 9.9500e-01,  ..., 5.0505e-05, 5.0505e-05,
         5.0505e-05],
        ...,
        [5.0505e-05, 5.0505e-05, 5.0505e-05,  ..., 9.9500e-01, 5.0505e-05,
         5.0505e-05],
        [5.0505e-05, 5.0505e-05, 5.0505e-05,  ..., 5.0505e-05, 9.9500e-01,
         5.0505e-05],
        [5.0505e-05, 5.0505e-05, 5.0505e-05,  ..., 5.0505e-05, 5.0505e-05,
         9.9500e-01]], device='cuda:0')


Epoch 1/200 | Loss: 3.8361 | Test Acc: 0.1551 | Top-5 Test Acc: 0.4136


Epoch 2/200 | Loss: 3.0666 | Test Acc: 0.2281 | Top-5 Test Acc: 0.5303


Epoch 3/200 | Loss: 2.5018 | Test Acc: 0.3474 | Top-5 Test Acc: 0.6752


Epoch 4/200 | Loss: 2.1276 | Test Acc: 0.3841 | Top-5 Test Acc: 0.7198


...

Epoch 196/200 | Loss: 1.4269 | Test Acc: 0.7906 | Top-5 Test Acc: 0.9420


Epoch 197/200 | Loss: 1.4270 | Test Acc: 0.7898 | Top-5 Test Acc: 0.9410


Epoch 198/200 | Loss: 1.4269 | Test Acc: 0.7898 | Top-5 Test Acc: 0.9424


Epoch 199/200 | Loss: 1.4268 | Test Acc: 0.7898 | Top-5 Test Acc: 0.9421


Epoch 200/200 | Loss: 1.4269 | Test Acc: 0.7899 | Top-5 Test Acc: 0.9425

ECE: 0.1993749588727951
NLL: 1.1601320505142212
Top-1 Test Acc: 0.7899 | Top-5 Test Acc: 0.9425

Model saved successfully to /content/drive/My Drive/MyModels/ls_02_0.pth



In [ ]:
from google.colab import runtime
runtime.unassign()